# 19 · HMDA：让模型更「省数据」——CatBoost 训练方式筛选（Colab GPU，五折）

**为什么做**：17 的学习曲线显示模型**受数据量限制**（训练数据 ×1.25 → AUC +0.0012，5/5 折）；18 显示伪标签几乎无效（+0.00008，3/5 折）——新的**真实**标签已经没有了。剩下的方向是让模型在同样的数据上**更稳、方差更小**。我们之前所有模型都只用了 GPU 默认的训练方式（对称树、Plain 提升、Cosine 打分、128 个分箱），以下选项**从未试过**：

|变体|改动|依据|
|---|---|---|
|`V1_ordered`|`boosting_type='Ordered'`|CatBoost 为小数据设计的有序提升，减少梯度估计的偏差；超过 5 万行时 GPU 默认关闭|
|`V2_newtonL2`|`score_function='NewtonL2'`|用二阶信息选择分裂（默认 Cosine）|
|`V3_depthwise`|逐层生长，深度 7，叶子至少 50 行|非对称树，叶子大小有下限|
|`V4_lossguide`|按损失逐叶生长，最多 48 个叶子，叶子至少 50 行|类似 LightGBM 的生长方式，但保留 CatBoost 的类别编码|
|`V5_poisson`|Poisson 抽样（仅 GPU）|另一种随机化|
|`V6_sub066`|行抽样 0.66（现为 0.8）|更强的随机化|
|`V7_border254`|254 个数值分箱（GPU 默认 128）|更细的数值切分|
|`V8_ctr1`|`max_ctr_complexity=1`|不做类别组合，降低类别编码的方差|
|`V9_drop_demo`|删去 10 个与 `derived_*` 重复的人口统计列|减少噪声列|

**协议**：特征 = M3 的 68 列；基线 `V0_base` ≈ M3（depth 6、L2=10、Bernoulli 0.8），学习率提高到 0.06 以加快筛选。每折在外层训练部分内留出 15% 早停，**直接用早停模型给外层验证打分**（不重训，所有变体相同，比较公平且省一半时间）。同一次运行、同一折配对比较。

**判读**：五折平均 ≥ +0.0005 且 ≥ 4/5 折才算有效；有效的变体之后用完整协议确认并做全量训练（20）。

**操作**：Colab 选 **T4 GPU**，从上到下运行。开头的「预检」会在一分钟内发现 GPU 不支持的参数并自动跳过该变体。预计约 1.5–2.5 小时（有序提升较慢）；每次训练后立即保存，断线后从头运行可续跑。

## 1. 环境、GPU 与配置

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'HMDA_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis',
)).expanduser().resolve()
PREPARED_DIR = PROJECT_DIR / 'artifacts' / 'prepared'
SEED = 42
CATBOOST_VERSION = '1.2.10'
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启 Colab 运行时后从头运行')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'catboost=={CATBOOST_VERSION}',
    ])
    importlib.invalidate_caches()

import catboost
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('未找到 nvidia-smi：请在 Colab 更改运行时类型，选择 T4 GPU 后从头运行')
gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True, capture_output=True, check=False,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('NVIDIA GPU 不可用：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = gpu_probe.stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'data_efficiency_screen_v1'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None                 # 例如 ['V0_base', 'V1_ordered']；None 表示全部
INNER_STOP_FRACTION = 0.15
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'gpu_tuning' / EXPERIMENT_ID

NO_FREQ = ['dti', 'ltv', 'income', 'missing']
LEAN = NO_FREQ + ['precision', 'thresholds']        # M3 的 68 列特征
DEMO_DUPLICATES = [                                 # 与 derived_* 重复的人口统计原始编码与「观察」标记
    'applicant_ethnicity-1', 'co-applicant_ethnicity-1',
    'applicant_race-1', 'co-applicant_race-1',
    'applicant_ethnicity_observed', 'co-applicant_ethnicity_observed',
    'applicant_race_observed', 'co-applicant_race_observed',
    'applicant_sex_observed', 'co-applicant_sex_observed',
]
SCREEN_BASE = dict(              # 接近 M3（C11）但学习率加倍以加快筛选；外层验证只评分
    depth=6, learning_rate=0.06, l2_leaf_reg=10,
    bootstrap_type='Bernoulli', subsample=0.8,
    loss_function='Logloss', eval_metric='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    iterations=6000, od_type='Iter', od_wait=200, use_best_model=True,
    allow_writing_files=False, verbose=False,
)
VARIANTS = {
    'V0_base': {},
    'V1_ordered': {'boosting_type': 'Ordered'},            # 超过 5 万行时 GPU 默认 Plain，从未试过
    'V2_newtonL2': {'score_function': 'NewtonL2'},         # 默认 Cosine
    'V3_depthwise': {'grow_policy': 'Depthwise', 'depth': 7, 'min_data_in_leaf': 50},
    'V4_lossguide': {'grow_policy': 'Lossguide', 'depth': 10, 'max_leaves': 48,
                     'min_data_in_leaf': 50, 'score_function': 'NewtonL2'},
    'V5_poisson': {'bootstrap_type': 'Poisson', 'subsample': 0.8},   # 仅 GPU 支持
    'V6_sub066': {'subsample': 0.66},                      # 更强的行抽样
    'V7_border254': {'border_count': 254},                 # GPU 默认 128 个数值分箱
    'V8_ctr1': {'max_ctr_complexity': 1},                  # 不做类别组合，降低方差
    'V9_drop_demo': {'drop': DEMO_DUPLICATES},             # 删去 10 个重复的人口统计列
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(VARIANTS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知变体：{sorted(unknown)}')
print(f'项目：{PROJECT_DIR}')
print(f'GPU：{GPU_NAME}；CatBoost {catboost.__version__}；折：{FOLDS}；输出：{OUTPUT_DIR}')

Mounted at /content/drive
项目：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis
GPU：Tesla T4, 15360 MiB；CatBoost 1.2.10；折：[1, 2, 3, 4, 5]；输出：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis/artifacts/gpu_tuning/data_efficiency_screen_v1


## 2. 回读并核对 01 的数据契约（与 04 相同）

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_hmda_data_preparation.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = PROJECT_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(PROJECT_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    PROJECT_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    PROJECT_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

数据核对通过：训练 105,000，测试 45,000，56 个原始特征。


accepted,0,1
fold,,
1,3151,17849
2,3151,17849
3,3151,17849
4,3151,17849
5,3152,17848


## 3. DTI 上下界（与 04 相同）

In [3]:
DTI_BOUNDS = {
    **{str(value): (float(value), float(value)) for value in range(36, 50)},
    '20%-<30%': (20.0, 30.0),
    '30%-<36%': (30.0, 36.0),
    '50%-60%': (50.0, 60.0),
    '<20%': (np.nan, 20.0),
    '>60%': (60.0, np.nan),
    'MISSING': (np.nan, np.nan),
}


def add_dti_bounds(frame):
    unknown = set(frame['debt_to_income_ratio'].unique()) - set(DTI_BOUNDS)
    require(not unknown, f'DTI 出现未知类别：{sorted(map(str, unknown))}')
    result = frame.copy()
    pairs = frame['debt_to_income_ratio'].map(DTI_BOUNDS)
    result['dti_lower'] = pairs.map(lambda pair: pair[0]).astype('float64')
    result['dti_upper'] = pairs.map(lambda pair: pair[1]).astype('float64')
    require(result.columns.tolist() == FEATURE_COLUMNS + ['dti_lower', 'dti_upper'],
            'DTI 特征列或顺序错误')
    require(result['debt_to_income_ratio'].equals(frame['debt_to_income_ratio']),
            '原始 DTI 类别被修改')
    return result


X_dti = add_dti_bounds(X_base)
X_test_dti = add_dti_bounds(X_test_base)
print(f'候选特征：原始 56 列 + 2 列 DTI 上下界 = {X_dti.shape[1]} 列。')
display(X_dti[['debt_to_income_ratio', 'dti_lower', 'dti_upper']].head())

候选特征：原始 56 列 + 2 列 DTI 上下界 = 58 列。


,debt_to_income_ratio,dti_lower,dti_upper
0,20%-<30%,20.0,30.0
1,49,49.0,49.0
2,20%-<30%,20.0,30.0
3,44,44.0,44.0
4,<20%,NaN,20.0


## 4. 第一轮特征块（与 10 相同）

In [4]:
def safe_ratio(numerator, denominator):
    valid = numerator.notna() & denominator.notna() & (denominator > 0)
    return (numerator / denominator.where(valid)).where(valid).astype('float64')


def build_blocks(train, test):
    both = pd.concat([train, test], ignore_index=True)
    blocks = {}
    blocks['dti'] = add_dti_bounds(both)[['dti_lower', 'dti_upper']]

    ltv_calc = 100 * safe_ratio(both['loan_amount'], both['property_value'])
    blocks['ltv'] = pd.DataFrame({
        'ltv_calc': ltv_calc,
        'ltv_gap': both['loan_to_value_ratio'] - ltv_calc,
    })

    income_dollars = both['income'] * 1000      # HMDA 收入单位为千美元
    blocks['income'] = pd.DataFrame({
        'income_to_area': safe_ratio(income_dollars, both['ffiec_msa_md_median_family_income']),
        'loan_to_income': safe_ratio(both['loan_amount'], income_dollars),
        'value_to_income': safe_ratio(both['property_value'], income_dollars),
    })

    missing = pd.DataFrame({
        'ltv': both['loan_to_value_ratio'].isna(),
        'pv': both['property_value'].isna(),
        'inc': both['income'].isna(),
        'term': both['loan_term'].isna(),
        'dti': both['debt_to_income_ratio'].eq('MISSING'),
        'age': both['applicant_age'].eq('MISSING'),
    })
    blocks['missing'] = pd.DataFrame({
        'missing_pattern': 'm' + missing.astype(int).astype(str).agg(''.join, axis=1),
        'missing_count': missing.sum(axis=1).astype('float64'),
    })

    freq = {}
    for column in ['county_code', 'derived_msa-md', 'state_code']:
        counts = both[column].map(both[column].value_counts()).astype('float64')
        freq[f'freq_{column}'] = counts.where(both[column].ne('MISSING'))
    blocks['freq'] = pd.DataFrame(freq)

    n_train = len(train)
    out = {}
    for name, frame in blocks.items():
        require(len(frame) == len(both), f'{name} 行数不符')
        out[name] = (frame.iloc[:n_train].reset_index(drop=True),
                     frame.iloc[n_train:].reset_index(drop=True))
    return out


BLOCKS = build_blocks(X_base, X_test_base)
NEW_CATEGORICAL = {'missing_pattern'}


def variant_frames(blocks):
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


for name, (train_block, _) in BLOCKS.items():
    described = train_block.describe(include='all').T
    print(f'\n[{name}] 缺失率：', train_block.isna().mean().round(3).to_dict())
    display(described.iloc[:, :6])


[dti] 缺失率： {'dti_lower': 0.117, 'dti_upper': 0.117}


,count,mean,std,min,25%,50%
dti_lower,92671.0,39.680947,10.654997,20.0,30.0,42.0
dti_upper,92730.0,41.033290,9.699260,20.0,36.0,40.0



[ltv] 缺失率： {'ltv_calc': 0.045, 'ltv_gap': 0.118}


,count,mean,std,min,25%,50%
ltv_calc,100244.0,82.070292,54.826068,0.302115,74.683544,88.235294
ltv_gap,92643.0,5.568211,439.329812,-7717.790000,-0.719136,0.000000



[income] 缺失率： {'income_to_area': 0.054, 'loan_to_income': 0.053, 'value_to_income': 0.088}


,count,mean,std,min,25%,50%
income_to_area,99343.0,1.563771,3.274112,-6.550349,0.746269,1.108830
loan_to_income,99468.0,2.891506,4.565102,0.002606,1.810560,2.692308
value_to_income,95792.0,3.713296,6.426108,0.009722,2.378641,3.288576



[missing] 缺失率： {'missing_pattern': 0.0, 'missing_count': 0.0}


,count,unique,top,freq,mean,std
missing_pattern,105000,42,m000000,88200,NaN,NaN
missing_count,105000.0,NaN,NaN,NaN,0.336019,0.956439



[freq] 缺失率： {'freq_county_code': 0.013, 'freq_derived_msa-md': 0.0, 'freq_state_code': 0.008}


,count,mean,std,min,25%,50%
freq_county_code,103661.0,364.801787,470.338073,1.0,67.0,200.0
freq_derived_msa-md,105000.0,4000.393038,7326.553843,2.0,353.0,927.0
freq_state_code,104176.0,6209.485976,4859.097575,2.0,2667.0,4441.0


## 5. 第二轮特征块（与 12 相同）

In [5]:
def read_raw_ltv(path):
    text = pd.read_csv(path, usecols=['loan_to_value_ratio'], dtype=str,
                       keep_default_na=False)['loan_to_value_ratio']
    return text


raw_ltv = pd.concat([read_raw_ltv(PREPARED_DIR / 'train_features.csv'),
                     read_raw_ltv(PREPARED_DIR / 'test_features.csv')], ignore_index=True)
both = pd.concat([X_base, X_test_base], ignore_index=True)
require(len(raw_ltv) == len(both), 'LTV 文本行数不符')
require(pd.to_numeric(raw_ltv.replace('', np.nan)).equals(both['loan_to_value_ratio'])
        or np.allclose(pd.to_numeric(raw_ltv.replace('', np.nan)), both['loan_to_value_ratio'],
                       equal_nan=True), 'LTV 文本与数值不一致')

decimals = raw_ltv.str.split('.').str[1].fillna('')
precision = np.select(
    [raw_ltv.eq(''), decimals.isin(['', '0']), decimals.str.len().eq(1),
     decimals.str.len().eq(2), decimals.str.len().eq(3)],
    ['missing', 'integer', 'dec1', 'dec2', 'dec3'], default='dec4plus',
)
ltv = both['loan_to_value_ratio']
band = np.select(
    [ltv.isna(), ltv < 80, ltv == 80, ltv < 95, ltv < 97, ltv < 100, ltv == 100],
    ['missing', 'lt80', 'eq80', '80_95', '95_97', '97_100', 'eq100'], default='gt100',
)
PROGRAM_MAX_LTV = {'1': 97.0, '2': 96.5, '3': 100.0, '4': 100.0}   # 常规/FHA/VA/RHS
unknown_types = set(both['loan_type']) - set(PROGRAM_MAX_LTV)
require(not unknown_types, f'未知 loan_type：{sorted(unknown_types)}')
implied_value = safe_ratio(both['loan_amount'] * 100, ltv)
term = both['loan_term']

round2 = {
    'precision': pd.DataFrame({'ltv_precision': precision}),
    'thresholds': pd.DataFrame({
        'ltv_band': band,
        'ltv_over_program': ltv - both['loan_type'].map(PROGRAM_MAX_LTV).astype('float64'),
    }),
    'implied': pd.DataFrame({
        'implied_property_value': implied_value,
        'implied_value_ratio': safe_ratio(implied_value, both['property_value']),
    }),
    'term': pd.DataFrame({
        'term_is_360': term.eq(360).astype('float64').where(term.notna()),
        'term_years': (term / 12).round(1),
    }),
}
n_train = len(X_base)
for name, frame in round2.items():
    require(len(frame) == len(both), f'{name} 行数不符')
    BLOCKS[name] = (frame.iloc[:n_train].reset_index(drop=True),
                    frame.iloc[n_train:].reset_index(drop=True))
NEW_CATEGORICAL |= {'ltv_precision', 'ltv_band'}

train_labels_check = y.to_numpy()
for column, frame in [('ltv_precision', round2['precision']), ('ltv_band', round2['thresholds'])]:
    table = pd.crosstab(frame[column].iloc[:n_train], train_labels_check, normalize='index')
    counts = frame[column].iloc[:n_train].value_counts()
    display(table.assign(rows=counts).round(3))
for name in ['implied', 'term']:
    display(BLOCKS[name][0].describe().T.round(3))
del both

col_0,0,1,rows
ltv_precision,,,
dec1,0.125,0.875,12131
dec2,0.156,0.844,9717
dec3,0.125,0.875,19531
dec4plus,0.154,0.846,306
integer,0.093,0.907,51441
missing,0.462,0.538,11874


col_0,0,1,rows
ltv_band,,,
80_95,0.122,0.878,14329
95_97,0.117,0.883,20177
97_100,0.129,0.871,6015
eq100,0.107,0.893,7295
eq80,0.081,0.919,13455
gt100,0.176,0.824,5711
lt80,0.096,0.904,26144
missing,0.462,0.538,11874


,count,mean,std,min,25%,50%,75%,max
implied_property_value,93126.0,526207.615,2740245.581,0.011,243523.316,367875.648,554999.602,6.270987e+08
implied_value_ratio,92643.0,1.051,2.960,0.003,0.989,1.000,1.009,1.108650e+02


,count,mean,std,min,25%,50%,75%,max
term_is_360,101693.0,0.841,0.366,0.0,1.0,1.0,1.0,1.0
term_years,101693.0,28.146,5.534,0.1,30.0,30.0,30.0,40.0


## 6. 特征选择与参数预检

In [6]:
lean_train, lean_test, lean_cats = variant_frames(LEAN)


def frames_for(spec):
    drop = spec.get('drop', [])
    unknown = set(drop) - set(lean_train.columns)
    require(not unknown, f'要删除的列不存在：{sorted(unknown)}')
    columns = [c for c in lean_train.columns if c not in drop]
    return lean_train[columns], [c for c in lean_cats if c not in drop]


def params_for(spec, **extra):
    params = dict(SCREEN_BASE)
    params.update({k: v for k, v in spec.items() if k != 'drop'})
    params.update(extra)
    return params


# 预检：每个变体在 3,000 行上训练 20 轮，几秒内暴露 GPU 不支持的参数组合，避免跑了一小时才报错
rng = np.random.default_rng(SEED)
probe_rows = rng.choice(len(y), 3000, replace=False)
failed = {}
for name, spec in VARIANTS.items():
    train, cats = frames_for(spec)
    try:
        model = catboost.CatBoostClassifier(**params_for(spec, iterations=20))
        model.fit(train.iloc[probe_rows[:2500]], y.iloc[probe_rows[:2500]], cat_features=cats,
                  eval_set=(train.iloc[probe_rows[2500:]], y.iloc[probe_rows[2500:]]))
        checked_probabilities(model, train.iloc[probe_rows[2500:]])
        print(f'预检通过：{name}（{train.shape[1]} 列）')
    except Exception as exc:
        failed[name] = repr(exc)
        print(f'预检失败，跳过：{name} → {exc!r}')
require('V0_base' not in failed, '基线 V0_base 预检失败，请检查环境')

预检通过：V0_base（68 列）
预检通过：V1_ordered（68 列）
预检通过：V2_newtonL2（68 列）
预检通过：V3_depthwise（68 列）
预检通过：V4_lossguide（68 列）
预检通过：V5_poisson（68 列）
预检通过：V6_sub066（68 列）
预检通过：V7_border254（68 列）
预检通过：V8_ctr1（68 列）
预检通过：V9_drop_demo（58 列）


## 7. 五折筛选（可续跑）

In [7]:
fixed_config(OUTPUT_DIR, {
    'purpose': 'data-efficiency screen on M3 features; inner 15% early stop, score probe model on outer fold',
    'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
    'screen_base': {k: v for k, v in SCREEN_BASE.items() if k != 'thread_count'},
    'features': LEAN, 'data_hashes': DATA_HASHES,
})
selected = {n: s for n, s in VARIANTS.items()
            if (RUN_ONLY is None or n in RUN_ONLY) and n not in failed}
for fold in FOLDS:
    fit_idx = np.flatnonzero(fold_assignment != fold)
    valid_idx = np.flatnonzero(fold_assignment == fold)
    inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION,
                                             random_state=SEED, stratify=y.iloc[fit_idx])
    for name, spec in selected.items():
        path = OUTPUT_DIR / f'{name}__fold{fold}.json'
        if path.is_file():
            saved = json.loads(path.read_text(encoding='utf-8'))
            require(saved['spec'] == spec, f'{name} 的定义已改变；请改变体名')
            print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
            continue
        train, cats = frames_for(spec)
        started = time.perf_counter()
        model = catboost.CatBoostClassifier(**params_for(spec))
        model.fit(train.iloc[inner_fit], y.iloc[inner_fit], cat_features=cats,
                  eval_set=(train.iloc[inner_stop], y.iloc[inner_stop]))
        best = int(model.get_best_iteration()) + 1
        probability = checked_probabilities(model, train.iloc[valid_idx])
        actual = y.iloc[valid_idx]
        pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                      'prob': probability}).to_csv(OUTPUT_DIR / f'{name}__fold{fold}__valid.csv',
                                                   index=False, float_format='%.12f')
        result = {'variant': name, 'fold': fold, 'spec': spec, 'feature_count': train.shape[1],
                  'best_iteration': best,
                  'hit_iteration_limit': best >= SCREEN_BASE['iterations'] - SCREEN_BASE['od_wait'],
                  'valid_auc': roc_auc_score(actual, probability),
                  'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
                  'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
                  'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        write_json(path, result)
        print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  最佳轮数={best}  '
              f'{result["seconds"]/60:.1f} 分钟', flush=True)
        del model
        gc.collect()

V0_base fold 1: AUC=0.911967  最佳轮数=1974  1.9 分钟
V1_ordered fold 1: AUC=0.912648  最佳轮数=3179  4.8 分钟
V2_newtonL2 fold 1: AUC=0.911207  最佳轮数=1031  1.1 分钟
V3_depthwise fold 1: AUC=0.911382  最佳轮数=612  0.3 分钟
V4_lossguide fold 1: AUC=0.911185  最佳轮数=281  0.3 分钟
V5_poisson fold 1: AUC=0.912814  最佳轮数=2077  2.0 分钟
V6_sub066 fold 1: AUC=0.912755  最佳轮数=1675  1.7 分钟
V7_border254 fold 1: AUC=0.912606  最佳轮数=1416  1.4 分钟
V8_ctr1 fold 1: AUC=0.912483  最佳轮数=1224  0.3 分钟
V9_drop_demo fold 1: AUC=0.912787  最佳轮数=1607  1.4 分钟
V0_base fold 2: AUC=0.926650  最佳轮数=2486  2.4 分钟
V1_ordered fold 2: AUC=0.926379  最佳轮数=4316  6.2 分钟
V2_newtonL2 fold 2: AUC=0.925906  最佳轮数=1219  1.3 分钟
V3_depthwise fold 2: AUC=0.925674  最佳轮数=574  0.3 分钟
V4_lossguide fold 2: AUC=0.924965  最佳轮数=423  0.3 分钟
V5_poisson fold 2: AUC=0.926031  最佳轮数=2513  2.3 分钟
V6_sub066 fold 2: AUC=0.926839  最佳轮数=1441  1.4 分钟
V7_border254 fold 2: AUC=0.926711  最佳轮数=2624  2.5 分钟
V8_ctr1 fold 2: AUC=0.925711  最佳轮数=1431  0.4 分钟
V9_drop_demo fold 2: AUC=0.926708

## 8. 与 V0_base 配对比较

`keep`：五折完成、平均提升 ≥ 0.0005 且至少 4/5 折优于 V0。

In [8]:
results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                        for p in sorted(OUTPUT_DIR.glob('*__fold?.json'))])
require(not results.empty, '还没有完成的结果')
auc = results.pivot(index='fold', columns='variant', values='valid_auc')
require('V0_base' in auc.columns, '缺少基线 V0_base')
delta = auc.sub(auc['V0_base'], axis=0)
table = pd.DataFrame({
    'folds_done': auc.notna().sum(),
    'mean_auc': auc.mean(),
    'mean_delta_vs_V0': delta.mean(),
    'folds_better_than_V0': (delta > 0).sum(),
    'mean_best_iteration': results.groupby('variant')['best_iteration'].mean(),
    'total_minutes': results.groupby('variant')['seconds'].sum() / 60,
})
table['keep'] = ((table['folds_done'] == 5) & (table['mean_delta_vs_V0'] >= 0.0005)
                 & (table['folds_better_than_V0'] >= 4))
table.loc['V0_base', 'keep'] = False
table = table.sort_values('mean_delta_vs_V0', ascending=False)
table.to_csv(OUTPUT_DIR / 'summary.csv')
display(table.round(6))
display(delta.round(6))
if failed:
    print('预检失败、未运行的变体：', failed)
kept = table.index[table['keep']].tolist()
print('达到保留标准（≥ +0.0005 且 ≥ 4/5 折）的变体：', kept if kept else '无')

,folds_done,mean_auc,mean_delta_vs_V0,folds_better_than_V0,mean_best_iteration,total_minutes,keep
variant,,,,,,,
V7_border254,5,0.918895,0.000230,4,1978.0,9.585734,False
V0_base,5,0.918665,0.000000,0,2034.2,9.778593,False
V1_ordered,5,0.918636,-0.000029,2,4013.0,29.403958,False
V5_poisson,5,0.918574,-0.000091,2,2498.8,11.670437,False
V6_sub066,5,0.918464,-0.000202,2,2001.2,9.633679,False
V9_drop_demo,5,0.918443,-0.000222,2,1920.8,7.937722,False
V3_depthwise,5,0.918121,-0.000544,1,613.6,1.444762,False
V8_ctr1,5,0.917979,-0.000686,1,1571.4,1.912846,False
V2_newtonL2,5,0.917846,-0.000819,0,1054.2,5.680483,False


variant,V0_base,V1_ordered,V2_newtonL2,V3_depthwise,V4_lossguide,V5_poisson,V6_sub066,V7_border254,V8_ctr1,V9_drop_demo
fold,,,,,,,,,,
1,0.0,0.000681,-0.000760,-0.000585,-0.000782,0.000847,0.000788,0.000639,0.000516,0.000820
2,0.0,-0.000270,-0.000744,-0.000975,-0.001684,-0.000618,0.000189,0.000061,-0.000938,0.000058
3,0.0,-0.000858,-0.000284,-0.000793,-0.000170,0.000152,-0.001284,0.000050,-0.000439,-0.000643
4,0.0,0.001434,-0.000548,0.000472,-0.000501,-0.000388,-0.000282,-0.000014,-0.002214,-0.001309
5,0.0,-0.001133,-0.001757,-0.000841,-0.002037,-0.000449,-0.000419,0.000414,-0.000353,-0.000038


达到保留标准（≥ +0.0005 且 ≥ 4/5 折）的变体： 无
